## Export S2-only model

In [19]:
!uv pip install onnx onnxscript

Using Python 3.13.9 environment at: /Users/taylor.denouden/Documents/PycharmProjects/kelp-o-matic/.venv
Resolved 10 packages in 221ms                                        
Prepared 2 packages in 1.81s                                             
Installed 4 packages in 291ms                               
 + ml-dtypes==0.5.4
 + onnx==1.20.1
 + onnx-ir==0.2.0
 + onnxscript==0.6.2


In [20]:
import io

import fsspec
import segmentation_models_pytorch as smp
import torch

EPS = 1e-10


class SKeMaModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = smp.Unet(
            encoder_name="tu-maxvit_tiny_tf_512",
            in_channels=10,
            encoder_weights=None,
        )

        self.register_buffer(
            "per_channel_mean",
            torch.tensor([
                2.08900522e02,
                2.70272557e02,
                1.52312422e02,
                9.87182507e02,
                3.26650321e02,
                5.65647882e-02,
                1.62913280e-01,
                -1.62913280e-01,
                1.90832012e07,
                1.09887867e-01,
            ]).view(1, -1, 1, 1),
        )

        self.register_buffer(
            "per_channel_std",
            torch.tensor([
                1.59021908e02,
                2.18393833e02,
                2.08355086e02,
                1.29667310e03,
                3.79794112e02,
                6.53314129e-01,
                7.11820223e-01,
                7.11820223e-01,
                2.07060299e10,
                3.90619966e-01,
            ]).view(1, -1, 1, 1),
        )

    def forward(self, x):
        # Unpack spectral bands
        blue = x.select(1, 0).unsqueeze(1)
        green = x.select(1, 1).unsqueeze(1)
        red = x.select(1, 2).unsqueeze(1)
        nir = x.select(1, 3).unsqueeze(1)
        re = x.select(1, 4).unsqueeze(1)

        # Compute vegetation indices
        ndvi = self.normalized_index(nir, red)
        gndvi = self.normalized_index(nir, green)
        ndvi_re = self.normalized_index(re, red)

        # Compute other indices
        ndwi = self.normalized_index(green, nir)
        chl_green = (nir / (green + EPS)) - 1  # Chlorophyll Index Green

        # Stack all bands and indices
        x_aug = torch.cat([blue, green, red, nir, re, ndvi, ndwi, gndvi, chl_green, ndvi_re], dim=1)

        x_aug_normalized = (x_aug - self.per_channel_mean) / self.per_channel_std

        return self.model(x_aug_normalized)

    @staticmethod
    def normalized_index(a, b):
        return (a - b) / (a + b + EPS)


model = SKeMaModel()

sample_input = torch.rand((2, 5, 512, 512), device=torch.device("cpu"), requires_grad=False)
model(sample_input)

tensor([[[[-0.5922, -0.6418,  0.3822,  ...,  0.5798, -0.8940, -0.2601],
          [-0.2932, -0.1471,  0.3270,  ...,  1.4800,  0.7815,  0.7526],
          [ 0.5656,  0.7844,  0.4758,  ...,  0.5334, -0.0974,  0.8997],
          ...,
          [-0.5788, -0.5244,  0.7161,  ...,  0.4853,  0.1908,  0.1119],
          [ 0.0479, -0.6125,  0.6306,  ...,  1.0703, -0.0043, -0.1709],
          [ 0.7121,  0.9889,  0.8124,  ...,  0.3788, -0.0873, -0.5510]]],


        [[[-0.4959, -0.5677, -0.1468,  ..., -0.4293, -0.4298, -0.2765],
          [-0.2326,  0.0167,  0.6265,  ...,  0.2460,  0.5083,  0.4256],
          [ 0.5613,  0.2503,  0.4402,  ..., -0.4827,  0.7994,  0.5329],
          ...,
          [-0.9346,  0.5549,  1.0778,  ...,  1.6986,  1.2045, -0.4636],
          [-0.0064, -0.6880,  1.7467,  ...,  1.6700,  0.6942,  0.1177],
          [ 0.5667,  0.7611,  0.9250,  ..., -0.1918, -0.5030, -0.2924]]]],
       grad_fn=<ConvolutionBackward0>)

In [21]:
with fsspec.open("https://huggingface.co/m5ghanba/SKeMa/resolve/main/modelS2Only.pth") as f:
    bytes_io = io.BytesIO(f.read())
state_dict = torch.load(bytes_io, map_location="cpu")

In [22]:
# Update keys
del state_dict["mean"]
del state_dict["std"]
model.load_state_dict(state_dict, strict=False)
model.eval()

SKeMaModel(
  (model): Unet(
    (encoder): TimmUniversalEncoder(
      (model): FeatureListNet(
        (stem): Stem(
          (conv1): Conv2dSame(10, 64, kernel_size=(3, 3), stride=(2, 2))
          (norm1): BatchNormAct2d(
            64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): GELUTanh()
          )
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (stages_0): MaxxVitStage(
          (blocks): Sequential(
            (0): MaxxVitBlock(
              (conv): MbConvBlock(
                (shortcut): Downsample2d(
                  (pool): AvgPool2dSame(kernel_size=(2, 2), stride=(2, 2), padding=(0, 0))
                  (expand): Identity()
                )
                (pre_norm): BatchNormAct2d(
                  64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
                  (drop): Identity()
                  (act): Identity()
  

In [23]:
torch.onnx.export(
    model,
    sample_input,
    "./Unet_tu-maxvit_tiny_tf_512_20260222.onnx",
    input_names=["input"],
    output_names=["output"],
    export_params=True,
    external_data=False,  # Store model weights in the model file
    opset_version=18,  # ONNX opset version
    do_constant_folding=True,  # Optimize constants
    verbose=False,
    # dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    dynamic_shapes={"x": (torch.export.Dim("batch"), 5, 512, 512)},
    dynamo=True,
)

/Users/taylor.denouden/.local/share/uv/python/cpython-3.13.9-macos-aarch64-none/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Applied 105 of general pattern rewrite rules.


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[batch,5,512,512]>
            ),
            outputs=(
                %"output"<FLOAT,[batch,1,512,512]>
            ),
            initializers=(
                %"model.encoder.model.stem.conv1.weight"<FLOAT,[64,10,3,3]>{Tensor(...)},
                %"model.encoder.model.stem.conv1.bias"<FLOAT,[64]>{Tensor(...)},
                %"model.encoder.model.stem.conv2.weight"<FLOAT,[64,64,3,3]>{TorchTensor(...)},
                %"model.encoder.model.stem.conv2.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"model.encoder.model.stages_0.blocks.0.conv.pre_norm.weight"<FLOAT,[64]>{TorchTensor(...)},
                %"model.encoder.model.stages_0.blocks.

## Export S2 + Bathymetry + Substrate model

In [4]:
EPS = 1e-10


class SKeMaBathyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = smp.Unet(
            encoder_name="tu-maxvit_tiny_tf_512",
            in_channels=12,
            encoder_weights=None,
        )

        self.register_buffer(
            "per_channel_mean",
            torch.tensor([
                2.02127847e02,
                2.64991799e02,
                1.45913497e02,
                9.57456953e02,
                3.20302883e02,
                1.37548690e00,
                -3.29322396e-01,
                3.47405153e01,
                3.66107438e-02,
                1.84492036e-01,
                -1.84492036e-01,
                2.22893410e07,
                9.99078992e-02,
            ]).view(1, -1, 1, 1),
        )

        self.register_buffer(
            "per_channel_std",
            torch.tensor([
                1.61504107e02,
                2.22303637e02,
                2.03997451e02,
                1.26105656e03,
                3.79069759e02,
                1.36767747e00,
                2.02345453e02,
                2.60894243e01,
                6.71879776e-01,
                7.23202999e-01,
                7.23202999e-01,
                2.25708723e10,
                4.06695959e-01,
            ]).view(1, -1, 1, 1),
        )

    def forward(self, x):
        # Unpack spectral bands
        blue = x.select(1, 0).unsqueeze(1)
        green = x.select(1, 1).unsqueeze(1)
        red = x.select(1, 2).unsqueeze(1)
        nir = x.select(1, 3).unsqueeze(1)
        re = x.select(1, 4).unsqueeze(1)
        substrate = x.select(1, 5).unsqueeze(1)
        bathymetry = x.select(1, 6).unsqueeze(1)

        # Compute vegetation indices
        ndvi = self.normalized_index(nir, red)
        gndvi = self.normalized_index(nir, green)
        ndvi_re = self.normalized_index(re, red)

        # Compute other indices
        ndwi = self.normalized_index(green, nir)
        chl_green = (nir / (green + EPS)) - 1  # Chlorophyll Index Green

        # Stack all bands and indices
        x_aug = torch.cat(
            [blue, green, red, nir, re, substrate, bathymetry, ndvi, ndwi, gndvi, chl_green, ndvi_re], dim=1
        )

        x_aug_normalized = (x_aug - self.per_channel_mean) / self.per_channel_std

        return self.model(x_aug_normalized)

    @staticmethod
    def normalized_index(a, b):
        return (a - b) / (a + b + EPS)


model = SKeMaBathyModel()

sample_input = torch.rand((2, 7, 512, 512), device=torch.device("cpu"), requires_grad=False)
model(sample_input)

tensor([[[[ 0.4813,  0.0931,  0.0299,  ...,  1.1187,  1.0073,  0.2813],
          [ 0.5153, -0.0539,  0.6094,  ...,  1.4073,  1.8217,  0.7350],
          [ 0.7110, -0.1710,  0.7305,  ...,  3.1022,  2.5016,  0.0418],
          ...,
          [ 0.2322, -0.3572, -0.6058,  ...,  0.0209,  0.8186,  0.3770],
          [ 0.7268,  0.2270, -0.0429,  ..., -0.0917,  1.0067,  0.5032],
          [-0.2261,  0.0787,  0.0792,  ...,  0.1037,  0.0944,  0.1754]]],


        [[[ 0.3767,  0.7756,  0.5304,  ...,  0.7425,  1.2530,  0.4901],
          [-0.0702, -0.5960,  0.2289,  ...,  0.7328,  1.0064,  1.0822],
          [ 0.5095, -0.2555, -0.3810,  ..., -0.4529,  0.7583,  1.3517],
          ...,
          [ 1.3123,  0.5650,  0.9073,  ...,  0.8774,  0.4388,  0.0916],
          [ 0.7773,  1.8775,  1.1773,  ...,  0.6576,  0.6688, -0.2499],
          [ 0.2814, -0.4011, -0.5158,  ...,  0.3432,  0.0538, -0.2632]]]],
       grad_fn=<ConvolutionBackward0>)

In [5]:
ckpt = torch.load("./Unet_tu-maxvit_tiny_tf_512_20250714_222203.ckpt", map_location="cpu")
state_dict = ckpt["state_dict"]

# Update keys
del state_dict["mean"]
del state_dict["std"]
model.load_state_dict(state_dict, strict=False)
model.eval()

SKeMaBathyModel(
  (model): Unet(
    (encoder): TimmUniversalEncoder(
      (model): FeatureListNet(
        (stem): Stem(
          (conv1): Conv2dSame(12, 64, kernel_size=(3, 3), stride=(2, 2))
          (norm1): BatchNormAct2d(
            64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): GELUTanh()
          )
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
        (stages_0): MaxxVitStage(
          (blocks): Sequential(
            (0): MaxxVitBlock(
              (conv): MbConvBlock(
                (shortcut): Downsample2d(
                  (pool): AvgPool2dSame(kernel_size=(2, 2), stride=(2, 2), padding=(0, 0))
                  (expand): Identity()
                )
                (pre_norm): BatchNormAct2d(
                  64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
                  (drop): Identity()
                  (act): Identity

In [6]:
torch.onnx.export(
    model,
    sample_input,
    "./Unet_tu-maxvit_tiny_tf_512_20250714_222203.onnx",
    input_names=["input"],
    output_names=["output"],
    export_params=True,
    external_data=False,  # Store model weights in the model file
    opset_version=15,  # ONNX opset version
    do_constant_folding=True,  # Optimize constants
    verbose=False,
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    # dynamic_shapes={"x": (torch.export.Dim("batch"), 5, 512, 512)},
    dynamo=False,
)

/var/folders/65/rp91cc952vq9zbnz_h9j_f6h0000gp/T/ipykernel_97418/2895710221.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
